In [ ]:
import pandas as pd
import commons as c
import numpy as np

from sklearn.metrics import f1_score, accuracy_score
import plotly.graph_objects as go
import plotly.express as px

# Boxplots of mutant detectability (distance between mutant and original)


In [ ]:
csv_path = 'results/dataframes/results_all_mutants.csv'
df = pd.read_csv(csv_path, dtype=c.type_dict)

In [ ]:
df.head()

# Plot Acc/F1 as function of Threshold

In [ ]:
def _getModelTolerance(model):
    if model == 'brisbane':
        tolerance_values_noisy = {
            'fidelity': 0.8238192155121457,
            'trace': 0.07095604148385458,
            'hellinger': 0.3664274445500234,
            'jensenshannon': 0.3107573573878005,
            'expectation': 0.27554319035115277
        }

    elif model == 'sherbrooke':
        tolerance_values_noisy = {
            'fidelity': 0.6610360085031542,
            'trace': 0.1438716383534881,
            'hellinger': 0.42870613054293644,
            'jensenshannon': 0.3644782707803318,
            'expectation': 0.25819115893612854
        }
    elif model == 'kyiv':
        tolerance_values_noisy = {
            'fidelity': 0.8046289596395843,
            'trace': 0.0897356767365067,
            'hellinger': 0.3282065923832697,
            'jensenshannon': 0.27689560267890234,
            'expectation': 0.18031809796093473
        }
    else:
        tolerance_values_noisy = {}

    return tolerance_values_noisy


def get_tolerance_values_87625(threshold, model=None):

    # Define tolerance values
    if threshold == 'I':
        tolerance_values = {
            'fidelity': 1 - 1e-14,
            'trace': 1e-13,
            'hellinger': 0.0708644302637939,
            'jensenshannon': 0.06453794560119465,
            'expectation': 0.0143011691815548
    }
    elif threshold == 'N':
        if model == None:
            raise ValueError("For threshold 'N', the 'model' parameter must be specified.")
        tolerance_values = _getModelTolerance(model)
    elif threshold == 'M':
        tolerance_values = {
            'fidelity': 0.922280490724269,
            'trace': 0.030780578765899045,
            'hellinger': 0.16151663844158012,
            'jensenshannon': 0.14104709915640773,
            'expectation': 0.03152815709501091
        }

    elif threshold == 'A':
        tolerance_values = {
            'fidelity': 0.6314291204717761,
            'trace': 0.1330880227046203,
            'hellinger': 0.4198099735344393,
            'jensenshannon': 0.3609924713920217,
            'expectation': 0.10472848966130349
        }
    else:
        raise ValueError(f"Invalid threshold: {threshold}.")

    return tolerance_values

In [ ]:
def find_best_thresholds(df, output_folder,  hw, m, distance='noisy_distance', start=0, end=1, steps=100, plot=True):
    thresholds = np.linspace(start, end, steps)
    f1_scores = []
    acc_scores = []

    best_thresh_f1 = None
    best_thresh_acc = None
    best_f1 = -1
    best_acc = -1

    # Convert 'equivalent' to 1 and 'non-equivalent' to 0
    y_true = df['nature'].map({'Equivalent mutant': 1, 'Non-Equivalent mutant': 0}).values

    for t in thresholds:
        y_pred = (df[distance] >= t).astype(int) if m == 'F' else (df[distance] <= t).astype(int)
        f1 = f1_score(y_true, y_pred)
        acc = accuracy_score(y_true, y_pred)
        f1_scores.append(f1)
        acc_scores.append(acc)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh_f1 = t
        if acc > best_acc:
            best_acc = acc
            best_thresh_acc = t

    if plot:
        fig = go.Figure()
        colors = px.colors.qualitative.Prism

        fig.add_trace(go.Scatter(
            x=thresholds, y=acc_scores,
            mode='lines', name='Accuracy',
            line=dict(color=colors[6])
        ))

        fig.add_trace(go.Scatter(
            x=thresholds, y=f1_scores,
            mode='lines', name='F1 Score',
            line=dict(color=colors[7])
        ))

        # Add vertical lines using Scatter to include them in the legend
        for x_val, name, color in [
            (best_thresh_acc, f"Best Acc: ({best_thresh_acc:.3f}, {best_acc:.3f})", colors[6]),
            (best_thresh_f1, f"Best F1: ({best_thresh_f1:.3f}, {best_f1:.3f})", colors[7])
        ]:
            fig.add_trace(go.Scatter(
                x=[x_val, x_val], y=[0, 1],
                mode='lines',
                line=dict(color=color, dash="solid"),
                name=name,
                showlegend=True
            ))

        # Add tolerance thresholds 
        i = 4
        for t in c.thresholds:  
            threshold_value = c.get_tolerance_values(t, hw)[c.metrics.get(m)]
            fig.add_trace(go.Scatter(
                x=[threshold_value, threshold_value], y=[0, 1],
                mode='lines',
                line=dict(color=colors[i], dash='dot'),
                name=f"{t} - 0.875 quantile",
                showlegend=True
            ))
            
            threshold_value = get_tolerance_values_87625(t, hw)[c.metrics.get(m)]
            fig.add_trace(go.Scatter(
                x=[threshold_value, threshold_value], y=[0, 1],
                mode='lines',
                line=dict(color=colors[i], dash='dashdot'),
                name=f"{t} - 0.87625 quantile",
                showlegend=True
            ))
            i -= 1

        fig.update_layout(
            xaxis_title="Threshold",
            yaxis_title="Score",
            xaxis_range=[start, end]
        )

        file_name = f"{hw}_acc_f1_{m}_500_steps"
        c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[0, 1], height=700)

    return best_thresh_f1, best_f1, best_thresh_acc, best_acc


In [ ]:
metrics = {'H': (0.05,0.42), 'J': (0.05,0.37), 'T': (0,0.22), 'F': (0.5,1), 'E': (0.012,0.6)}
output_folder='results/test_thresholds/quantiles/'

for hw in c.hardware:
    print(f"===================== {hw} =======================")
    df_hw = df[df['hardware'] == hw]
    for m, (start,end) in metrics.items():
        print(f"===================== {m} =======================")
        df_metric = df_hw[df_hw['metric'] == m]
        best_thresh_f1, best_f1, best_thresh_acc, best_acc = find_best_thresholds(df_metric, output_folder, hw, m, start=start, end=end, steps=500)
        print(f"Metric: {m}, Best threshold (according to Accuracy): {best_thresh_acc:.4f}, Best threshold (according to F1 score): {best_thresh_f1:.4f}")
        #print(f"Metric: {m}, Best threshold: {best_thresh_acc:.4f}, Best Accuracy: {best_acc:.4f}")
        #print(f"Metric: {m}, Best threshold: {best_thresh_f1:.4f}, Best F1 score: {best_f1:.4f}")

## Noiseless threshold

In [ ]:
#quantiles = [0.5, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
quantiles = [0.87625, 0.875, 0.876, 0.8763, 0.877]

# H - 0.0701
# J - 0.0641 ~ 0.0661
# E - 0.0261 ~ 0.0281 ~ 0.0341

def tvalues(threshold):

    # Define tolerance values
    if threshold == 0.5:
        tolerance_values = {
            'fidelity': 1 - 1e-14,
            'trace': 1e-13,
            'hellinger': 0.04167558253647273,
            'jensenshannon': 0.040462095686274235,
            'expectation': 0.012071688887134691
        }
    elif threshold == 0.7:
        tolerance_values = {
            'fidelity': 1 - 1e-14,
            'trace': 1e-13,
            'hellinger': 0.058509535599050685,
            'jensenshannon': 0.056859406171761696,
            'expectation': 0.01312460727172336
        }
    elif threshold == 0.75:
        tolerance_values = {
            'fidelity': 1 - 1e-14,
            'trace': 1e-13,
            'hellinger': 0.05999462573410876,
            'jensenshannon': 0.0573756781781595,
            'expectation': 0.01341994679081911
        }
    elif threshold == 0.8:
        tolerance_values = {
            'fidelity': 1 - 1e-14,
            'trace': 1e-13,
            'hellinger': 0.061307822376336335,
            'jensenshannon': 0.058293542340380554,
            'expectation': 0.013727422344562933
        }
    elif threshold == 0.85:
        tolerance_values = {
            'fidelity': 1 - 1e-14,
            'trace': 1e-13,
            'hellinger': 0.0629078709744999,
            'jensenshannon': 0.059248561758507276,
            'expectation': 0.014066926738899243
        }
        
    elif threshold == 0.875:
        tolerance_values = {
            'fidelity': 1 - 1e-14,
            'trace': 1e-13,
            'hellinger': 0.06561412848403149,
            'jensenshannon': 0.06045268171304313,
            'expectation': 0.014284606173994616
        }
        
    elif threshold == 0.8763:
        tolerance_values = {
            'fidelity': 1 - 1e-14,
            'trace': 1e-13,
            'hellinger': 0.07576949109821934,
            'jensenshannon': 0.0680261048773476,
            'expectation': 0.014302487079505054
        }
    elif threshold == 0.87625:
        tolerance_values = {
            'fidelity': 1 - 1e-14,
            'trace': 1e-13,
            'hellinger': 0.0708644302637939,
            'jensenshannon': 0.06453794560119465,
            'expectation': 0.0143011691815548
        }
    elif threshold == 0.876:
        tolerance_values = {
            'fidelity': 1 - 1e-14,
            'trace': 1e-13,
            'hellinger': 0.06665324915086634,
            'jensenshannon': 0.061325456667957634,
            'expectation': 0.014297687625886919
        }
    elif threshold == 0.877:
        tolerance_values = {
            'fidelity': 1 - 1e-14,
            'trace': 1e-13,
            'hellinger': 0.12226387182415822,
            'jensenshannon': 0.10547637462415897,
            'expectation': 0.014308844608683324
        }
    else:
        raise ValueError(f"Invalid threshold: {threshold}. Must be one of {quantiles}")

    return tolerance_values


In [ ]:
def find_best_ideal_thresholds(df, output_folder,  hw, m, distance='noisy_distance', start=0, end=1, steps=100, plot=True):
    thresholds = np.linspace(start, end, steps)
    f1_scores = []
    acc_scores = []

    best_thresh_f1 = None
    best_thresh_acc = None
    best_f1 = -1
    best_acc = -1

    # Convert 'equivalent' to 1 and 'non-equivalent' to 0
    y_true = df['nature'].map({'Equivalent mutant': 1, 'Non-Equivalent mutant': 0}).values

    for t in thresholds:
        y_pred = (df[distance] >= t).astype(int) if m == 'F' else (df[distance] <= t).astype(int)
        f1 = f1_score(y_true, y_pred)
        acc = accuracy_score(y_true, y_pred)
        f1_scores.append(f1)
        acc_scores.append(acc)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh_f1 = t
        if acc > best_acc:
            best_acc = acc
            best_thresh_acc = t

    if plot:
        fig = go.Figure()
        colors = px.colors.qualitative.Prism

        fig.add_trace(go.Scatter(
            x=thresholds, y=acc_scores,
            mode='lines', name='Accuracy',
            line=dict(color='green')
        ))

        fig.add_trace(go.Scatter(
            x=thresholds, y=f1_scores,
            mode='lines', name='F1 Score',
            line=dict(color='blue')
        ))

        # Add vertical lines using Scatter to include them in the legend
        for x_val, name, color in [
            (best_thresh_acc, f"Best Acc: ({best_thresh_acc:.3f}, {best_acc:.3f})", 'green'),
            (best_thresh_f1, f"Best F1: ({best_thresh_f1:.3f}, {best_f1:.3f})", 'blue')
        ]:
            fig.add_trace(go.Scatter(
                x=[x_val, x_val], y=[0, 1],
                mode='lines',
                line=dict(color=color, dash="dash"),
                name=name,
                showlegend=True
            ))

        # Add tolerance thresholds 
        i = 4
        for t in quantiles:  
            threshold_value = tvalues(t)[c.metrics.get(m)]
            fig.add_trace(go.Scatter(
                x=[threshold_value, threshold_value], y=[0, 1],
                mode='lines',
                line=dict(color=colors[i], dash='dot'),
                name=f"{t}",
                showlegend=True
            ))
            i -= 1

        fig.update_layout(
            xaxis_title="Threshold",
            yaxis_title="Score",
            xaxis_range=[start, end]
        )

        file_name = f"{hw}_acc_f1_{m}_500_steps"
        c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[0, 1], height=700)

    return best_thresh_f1, best_f1, best_thresh_acc, best_acc


In [ ]:
metrics = {'H': (0,0.1), 'J': (0,0.1), 'E': (0,0.1)}
output_folder='results/test_thresholds/ideal/'

for hw in ["kyiv"]:
    print(f"===================== {hw} =======================")
    df_hw = df[df['hardware'] == hw]
    for m, (start,end) in metrics.items():
        print(f"===================== {m} =======================")
        df_metric = df_hw[df_hw['metric'] == m]
        best_thresh_f1, best_f1, best_thresh_acc, best_acc = find_best_ideal_thresholds(df_metric, output_folder, hw, m, distance='ideal_distance', start=start, end=end, steps=500)
        # print(f"Metric: {m}, Best threshold (according to Accuracy): {best_thresh_acc:.4f}, Best threshold (according to F1 score): {best_thresh_f1:.4f}")
        print(f"Metric: {m}, Best threshold (according to Accuracy): {best_thresh_acc}, Best threshold (according to F1 score): {best_thresh_f1}")
        #print(f"Metric: {m}, Best threshold: {best_thresh_acc:.4f}, Best Accuracy: {best_acc:.4f}")
        #print(f"Metric: {m}, Best threshold: {best_thresh_f1:.4f}, Best F1 score: {best_f1:.4f}")